## Wczytanie bibliotek i zbiorów

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                            classification_report)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
import joblib
data = joblib.load('../data/processed_data.pkl')
X_train_processed = data['X_train_processed']
X_val_processed = data['X_val_processed']
X_test_processed = data['X_test_processed']
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']
feature_names = data['feature_names']
X_train = data['X_train']

# 3. Trening i ewaluacja modeli

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_v, y_v, name):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_v)
    y_prob = model.predict_proba(X_v)[:, 1]
    return {
        'Model': name,
        'Accuracy':  np.round(accuracy_score(y_v, y_pred), 4),
        'Precision': np.round(precision_score(y_v, y_pred), 4),
        'Recall':    np.round(recall_score(y_v, y_pred), 4),
        'F1-Score':  np.round(f1_score(y_v, y_pred), 4),
        'ROC-AUC':   np.round(roc_auc_score(y_v, y_prob), 4),
    }

results = []
trained_models = {}

### Regresja logistyczna

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=123)
res = evaluate_model(lr, X_train_processed, y_train, X_val_processed, y_val, 'Logistic Regression')
results.append(res)
trained_models['Logistic Regression'] = lr

for k, v in res.items():
    print(f"{k}: {v}")

### Drzewo decyzyjne

In [ ]:
dt = DecisionTreeClassifier(random_state=123)
res = evaluate_model(
    dt,
    X_train_processed, y_train,
    X_val_processed, y_val,
    'Decision Tree'
)

results.append(res)
trained_models['Decision Tree'] = dt

for k, v in res.items():
    print(f"{k}: {v}")

### Random forest

In [ ]:
rf = RandomForestClassifier(random_state=123)

res = evaluate_model(
    rf,
    X_train_processed, y_train,
    X_val_processed, y_val,
    'Random Forest'
)

results.append(res)
trained_models['Random Forest'] = rf

for k, v in res.items():
    print(f"{k}: {v}")

### SVC

In [ ]:
sv = SVC(random_state=123, probability=True)

res = evaluate_model(
    sv,
    X_train_processed, y_train,
    X_val_processed, y_val,
    'SVC'
)

results.append(res)
trained_models['SVC'] = sv

for k, v in res.items():
    print(f"{k}: {v}")

In [ ]:
param_dist = {
    'n_estimators': [100, 150, 200, 250, 300, 400],
    'max_depth': [10, 15, 20, 25, 30, 35, 40, None],
    'min_samples_split': [2, 3, 5],
    'min_samples_leaf': [1, 2, 3]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=123),
    param_distributions=param_dist,
    n_iter=100,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    random_state=123
)

print('Szukanie najlepszych parametrów...')
random_search.fit(X_train_processed, y_train)

print('Najlepsze znalezione parametry:', random_search.best_params_)

best_rf = random_search.best_estimator_
res_tuned = evaluate_model(best_rf, X_train_processed, y_train, X_val_processed, y_val, 'Tuned Random Forest')

results.append(res_tuned)
trained_models['Tuned Random Forest'] = best_rf

print('\nWyniki modelu po tuningu na zbiorze walidacyjnym:')
for k, v in res_tuned.items():
    print(f'{k}: {v}')

## Porównanie modeli

In [ ]:
results_df = pd.DataFrame(results).set_index('Model')
print("Porównanie modeli na zbiorze walidacyjnym")
print(results_df)

print("\nNajlepszy model wg. metryki")
for col in results_df.columns:
    best = results_df[col].idxmax()
    print(f"  {col:12s}: {best} ({results_df.loc[best, col]:.4f})")


## Wybór najlepszego modelu i ewaluacja na zbiorze testowym (todo)

In [ ]:
final_model = trained_models['Tuned Random Forest']

y_test_pred = final_model.predict(X_test_processed)
y_test_prob = final_model.predict_proba(X_test_processed)[:, 1]

print("Wyniki na zbiorze testowym")
print(f"Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall:    {recall_score(y_test, y_test_pred):.4f}")
print(f"F1-Score:  {f1_score(y_test, y_test_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_test_prob):.4f}")

print(classification_report(y_test, y_test_pred))

In [ ]:
best_rf = trained_models['Tuned Random Forest']

feature_names = best_rf.feature_names_in_
importances = best_rf.feature_importances_

feature_importance_df = pd.DataFrame({
    'Cecha': feature_names,
    'Ważność': importances
})

feature_importance_df = feature_importance_df.sort_values(by='Ważność', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=feature_importance_df.head(25), x='Ważność', y='Cecha', palette='viridis')
plt.title('Top 25 najważniejszych cech (Random Forest)')
plt.xlabel('Ważność (wpływ na predykcję)')
plt.ylabel('Cecha')
plt.tight_layout()
plt.show()

### Zapisanie modelu

In [ ]:
joblib.dump(trained_models['Tuned Random Forest'], '../data/best_model.pkl')